In [43]:
import base64
import os.path
import re
from email.message import EmailMessage

import docx
import google.auth
import pandas as pd
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


In [44]:
account = 'panirand'
# account = "iumrs"

if account == "panirand":
    token_path = 'credentials/token_panirand.json'
    creds_path = 'credentials/credentials_panirand.json'
elif account == "iumrs":
    token_path = 'credentials/token_iumrs.json'
    creds_path = 'credentials/credentials_iumrs.json'
else:
    raise ValueError("Invalid account specified. Please choose 'panirand' or 'iumrs'.")

In [45]:
# If modifying these scopes, delete the file token.json.
# SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
# SCOPES = ["https://www.googleapis.com/auth/gmail.compose"]
SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]


def get_auth(token_path=token_path, creds_path=creds_path):
    """Shows basic usage of the Gmail API.
    Lists the user's Gmail labels.
    """
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                creds_path, SCOPES
            )
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(token_path, "w") as token:
            token.write(creds.to_json())

    try:
        # Call the Gmail API
        service = build("gmail", "v1", credentials=creds)
        results = service.users().labels().list(userId="me").execute()
        labels = results.get("labels", [])

        if not labels:
            print("No labels found.")
            return
        print("Labels:")
        for label in labels:
            print(label["name"])

    except HttpError as error:
        # TODO(developer) - Handle errors from gmail API.
        print(f"An error occurred: {error}")


get_auth()

Labels:
CHAT
SENT
INBOX
IMPORTANT
TRASH
DRAFT
SPAM
CATEGORY_FORUMS
CATEGORY_UPDATES
CATEGORY_PERSONAL
CATEGORY_PROMOTIONS
CATEGORY_SOCIAL
STARRED
UNREAD
Conversation History


In [46]:
df = pd.read_excel("output_excel/S01_data.xlsx")
df.head()

,Num,Symposia,Role,Thai,Name,Affiliation,Email,Ignore,name_strip,title,filename,docx,pdf
0,1,"Semiconductors, Photonic Materials and Devices",Chair,รองศาสตราจารย์ ดร. อนุชา วัชระภาสร,Assoc. Prof. Dr. Anucha Watcharapasorn,Chiang Mai University,anucha.w@cmu.ac.th,NaN,Anucha Watcharapasorn,Assoc. Prof. Dr.,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...,Invitation IUMRS-ICEM2026_Symp1_Chair_Anucha W...
1,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ศาสตราจารย์ ดร. วิษณุ เพชรภา,Prof. Dr. Wisanu Pecharapa,King Mongkut's Institute of Technology Ladkrabang,wisanu.pe@kmitl.ac.th,NaN,Wisanu Pecharapa,Prof. Dr.,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Wisan...
2,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ดร. ทศพร เลิศวณิชผล,Dr. Tossaporn Lertvanithphol,National Electronics and Computer Technology C...,tossaporn.ler@nectec.or.th,NaN,Tossaporn Lertvanithphol,Dr.,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Tossa...
3,1,"Semiconductors, Photonic Materials and Devices",Co-chair,รองศาสตราจารย์ ดร. ดลเดช ตันตระวิวัฒน์,Assoc. Prof. Dr. Doldet Tantraviwat,Chiang Mai University,doldet.tantraviwat@cmu.ac.th,NaN,Doldet Tantraviwat,Assoc. Prof. Dr.,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Dolde...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Dolde...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Dolde...
4,1,"Semiconductors, Photonic Materials and Devices",Co-chair,ผู้ช่วยศาสตราจารย์ ดร. ทุติยาภรณ์ ทิวาวงศ์.,Asst. Prof. Dr. Thutiyaporn Thiwawong,King Mongkut's Institute of Technology Ladkrabang,thutiyaporn.th@kmitl.ac.th,NaN,Thutiyaporn Thiwawong,Asst. Prof. Dr.,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Thuti...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Thuti...,Invitation IUMRS-ICEM2026_Symp1_Co-chair_Thuti...


In [47]:
filepath = f'output_docx_chair/{df["docx"].values[0]}'
doc = docx.Document(filepath)
# Joins all paragraphs with a newline

selected_text = []
for para in doc.paragraphs:
    text = para.text.strip()

    # Exclude paragraphs containing "Invitation to IUMRS-ICEM 2026" or "February 27, 2026"
    match_exclude = re.search(r'Invitation to IUMRS-ICEM 2026|February 27, 2026', text)
    if match_exclude:
        continue

    # Add a tab at the beginning of paragraphs containing "On behalf of", "We trust that", or "Regarding your trip"
    match_add_tab = re.search(r'On behalf of|We trust that|Regarding your trip', text)
    if match_add_tab:
        text = '\t' + text

    # Append the processed text to the list
    selected_text.append(text)

# Join the selected text with newlines
text_joined = '\n'.join(selected_text)

# Take the text starting from "Dear"
match_dear = re.search(r'Dear\s+.*?,', text_joined)
text_joined = text_joined[match_dear.start():]

# Replace multiple newlines with a maximum of two
text_joined = re.sub(r'\n\n\n+', '\n\n', text_joined) 
print(text_joined)

Dear Assoc. Prof. Dr. Anucha Watcharapasorn,

	On behalf of the Thailand Materials Research Society (MRS-Thailand), in partnership with the International Union of Materials Research Societies (IUMRS), we are pleased to invite you as a Session Chair for the Symposium 1: Semiconductors, Photonic Materials and Devices at our upcoming conference, the International Union of Materials Research Societies– the 19th International Conference on Electronic Materials (IUMRS-ICEM2026) to be held during June 28 – July 1, 2026 at Convention Center, the Empress Hotel, Chiang Mai, Thailand. For more conference details, please visit the conference website: https://www.iumrs-icem2026.org.
	We trust that your contributions and recognized expertise in the field would greatly enhance the scientific quality of the conference. As a Symposium Chair, your role may include reviewing the abstracts, organizing the session schedule, moderating the oral presentations and facilitating the discussion, including poster

In [48]:
def get_email_body_from_docx(file_path):
    doc = docx.Document(file_path)
    selected_text = []
    for para in doc.paragraphs:
        text = para.text.strip()

        # Exclude paragraphs containing "Invitation to IUMRS-ICEM 2026" or "February 27, 2026"
        match_exclude = re.search(r'Invitation to IUMRS-ICEM 2026|February 27, 2026', text)
        if match_exclude:
            continue

        # Add a tab at the beginning of paragraphs containing "On behalf of", "We trust that", or "Regarding your trip"
        match_add_tab = re.search(r'On behalf of|We trust that|Regarding your trip', text)
        if match_add_tab:
            text = '\t' + text

        # Append the processed text to the list
        selected_text.append(text)

    # Join the selected text with newlines
    text_joined = '\n'.join(selected_text)

    # Take the text starting from "Dear"
    match_dear = re.search(r'Dear\s+.*?,', text_joined)
    if match_dear:
        text_joined = text_joined[match_dear.start():]

    # Replace multiple newlines with a maximum of two
    text_joined = re.sub(r'\n\n\n+', '\n\n', text_joined) 
    return text_joined

In [49]:
if os.path.exists(token_path):
    creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    
service = build("gmail", "v1", credentials=creds)
message = EmailMessage()

filepath = f'output_docx_chair/{df["docx"].values[0]}'
email_body = get_email_body_from_docx(filepath)
message.set_content(email_body)

message["To"] = "gduser1@workspacesamples.dev"
message["From"] = "gduser2@workspacesamples.dev"
message["Subject"] = "Automated draft"

# encoded message
encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()

create_message = {"message": {"raw": encoded_message}}
# pylint: disable=E1101
draft = service.users().drafts().create(userId="me", body=create_message).execute()

print(f"Draft id: {draft['id']}\nDraft message: {draft['message']}")

Draft id: r1706444462611386964
Draft message: {'id': '19c9fe6709586005', 'threadId': '19c9fe6709586005', 'labelIds': ['DRAFT']}


In [50]:
if os.path.exists(token_path):
    creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    
    service = build("gmail", "v1", credentials=creds)
    message = EmailMessage()

    filepath = f'output_docx_chair/{df["docx"].values[0]}'
    email_body = get_email_body_from_docx(filepath)
    
    # Set plain text version (fallback)
    message.set_content(email_body)
    
    # Set HTML version with bold and links
    html_body = f"""
    <html>
      <body>
        <h2>Automated Draft</h2>
        <p><strong>{email_body}</strong></p>
        <p>Click <a href="https://example.com">here</a> for more details.</p>
      </body>
    </html>
    """
    message.add_alternative(html_body, subtype="html")

    message["To"] = "gduser1@workspacesamples.dev"
    message["From"] = "gduser2@workspacesamples.dev"
    message["Subject"] = "Automated draft"

    # encoded message
    encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()

    create_message = {"message": {"raw": encoded_message}}
    # pylint: disable=E1101
    draft = service.users().drafts().create(userId="me", body=create_message).execute()

    print(f"Draft id: {draft['id']}\nDraft message: {draft['message']}")

Draft id: r3812702845049969360
Draft message: {'id': '19c9feb18210cd81', 'threadId': '19c9feb18210cd81', 'labelIds': ['DRAFT']}
